## TODO

- [ ] Parse boolean fields explicitly instead of using `bool(value)`, which incorrectly treats strings such as `"false"` as `True`.
- [ ] Only convert floats to integers when they represent whole numbers; avoid truncating legitimate decimal values.
- [ ] Make `_im` array conversion tolerate whitespace and integer-like values such as `"1922.0"`.
- [ ] Handle a missing `"GeoJSON"` entry in `reference_uri_dict` without raising a `KeyError`.
- [ ] Validate `ID` before constructing filenames so missing values do not produce `nan_BL_Aardvark.json`.
- [ ] Replace the heuristic filename parsing with explicit ARK parsing to prevent filename collisions.
- [ ] Detect duplicate output filenames before files are overwritten.
- [ ] Decide how reruns should handle stale JSON files left in the output directory.
- [ ] Remove the unused `index` variable or restore a deliberate index-based fallback.
- [ ] Document that `dct_references_s` must be a JSON-encoded string inside the outer JSON document.
- [ ] Add validation and useful error messages for missing CSV columns, malformed identifiers, invalid numeric arrays, and absent input files.
- [ ] Define the input contract separately from generated fields: allow optional CSV overrides, but otherwise default Metadata Version to configurable `Aardvark`, generate Modified as the JSON-production timestamp in UTC, default Provider to `American Geographical Society Library – UWM Libraries`, and default Suppressed to boolean `false`.
- [ ] Decide where workflow defaults and their override precedence live (for example, a versioned configuration file plus CLI options); reject unknown configuration keys and record the effective configuration in conversion diagnostics.
- [ ] Reconcile the general-workshop `Alternative Title` column with the AGSL profile label `AGSL Call Number`; either standardize the template on the AGSL label or support an explicit, tested alias without silently dropping values.
- [ ] Treat Geometry as required by the current community JSON Schema. Define how it can be derived from source extents or Bounding Box, how raw west/south/east/north coordinates become `ENVELOPE(W,E,N,S)`, and how records with no derivable geometry are reported for review.
- [ ] Evaluate reusable bounding-box and controlled-value logic in `../Clean-Validate/04_clean-validate.ipynb`, but separate its raw `west,south,east,north` representation from Aardvark ENVELOPE ordering and replace silent corrections/defaults with logged, testable policy.
- [ ] Explore a reproducible Theme classification step using existing projects or controlled-vocabulary mappings; preserve human overrides and flag uncertain classifications rather than inventing them silently.
- [ ] Compare every local profile obligation with current OGM documentation, document intentional AGSL overrides and their practical GeoBlacklight/Solr impact, and otherwise align the profile to the community standard.
- [ ] Resolve the `gbl_dateRange_drsim` contradiction between the profile/schema array type and OGM guidance showing one bracketed string; verify actual GeoBlacklight/Solr behavior and raise an OGM community issue if the published sources remain inconsistent.
- [ ] Strip surrounding whitespace from every pipe-delimited value in Python, even when OpenRefine is used upstream, and test values such as `Index maps | Topographic Maps`.
- [ ] Preserve and test the critical ARK distinction: AGSL `id` uses the modified `ark:-77981-...` permalink form, while `dct_identifier_sm` uses normalized `ark:/77981/...`; never apply one normalization rule to both fields.
- [ ] Skip wholly blank CSV rows and test that they never create `nan_BL_Aardvark.json`; keep the blank template header-only.
- [ ] Warn or fail on unknown input columns instead of silently ignoring them, including accidental columns such as `Column1`; decide explicitly whether `Github View` is removed or mapped to a supported/local reference URI.
- [ ] Validate every generated record against `../../schema/geoblacklight-schema-aardvark.json` before final output and report all row/field errors together; update tests whenever the pinned community schema changes.
- [ ] Add optional NOID mint-and-bind support behind a testable service interface, preserving supplied IDs and making retries idempotent so the same source record cannot receive multiple ARKs.
- [ ] Add representative conversion tests covering defaults and overrides, booleans, both ARK forms, Unicode, trimmed arrays, missing and minted IDs, references, geometry derivation, schema failures, blank rows, unknown columns, and decimal values.


## Step 1. Import Modules

In [1]:
import pandas as pd
import json
import os

## Step 2. Specify the file paths

In [ ]:
csv_file_path = '../csv2JSON/OpenIndexMaps_Aardvark_workshop.csv'  # the name of your CSV
reference_uris_file_path = '../aardvark-profile/referenceURIs.csv'  # CSV mapping reference URIs and labels
full_schema_file_path = '../aardvark-profile/aardvark.csv'  # CSV mapping OGM Aardvark fields and labels
output_dir = 'json_output'  # Output directory

## Step 3. Define the function

In [ ]:
def convert_csv_to_json(csv_file_path, reference_uris_file_path, full_schema_file_path, output_dir):
    # Load the CSV data
    csv_data = pd.read_csv(csv_file_path)
    # Load the reference URIs data
    reference_uris_data = pd.read_csv(reference_uris_file_path)
    reference_uri_dict = dict(zip(reference_uris_data['LABEL'], reference_uris_data['URI']))
    # Load the full schema data
    full_schema_data = pd.read_csv(full_schema_file_path)
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Function to handle multivalued fields
    def split_multivalues(val, format):
        value_list = val.split('|') if pd.notna(val) and '|' in val else [val]
        if format == "im":
            value_list = [int(item) for item in value_list]
        return value_list
    
    def normalize_ark_identifier(value):
        if pd.isna(value):
            return value
        identifier = str(value).strip()
        if identifier.startswith('ark:-'):
            return identifier.replace('ark:-', 'ark:/', 1).replace('-', '/', 1)
        return identifier
    
    # Function to construct JSON data from a row
    def construct_json_data(row):
        json_data = {}
        for _, schema_row in full_schema_data.iterrows():
            label = schema_row['Label']
            field_name = schema_row['Field Name']
            field_type = schema_row['Field Type']
            
            if field_name in ["dct_references_s"]:  # Handle references separately
                references = {}
                for ref_label in reference_uri_dict.keys():
                    if pd.notna(row.get(ref_label)):
                        references[reference_uri_dict[ref_label]] = row[ref_label]
                if str(row.get("Format", "")).strip() == "GeoJSON" and pd.notna(row.get("Download")):
                    references[reference_uri_dict["GeoJSON"]] = row["Download"]
                if references:
                    json_data[field_name] = json.dumps(references)
            elif pd.notna(row.get(label)) or (label == "Identifier" and pd.notna(row.get("ID"))):
                source_value = row.get("ID") if label == "Identifier" and pd.notna(row.get("ID")) else row.get(label)
                if label == "Identifier":
                    source_value = normalize_ark_identifier(source_value)
                if field_type == "Array":
                    json_data[field_name] = split_multivalues(str(source_value), field_name[-2:])
                elif field_type == "Boolean or string":
                    json_data[field_name] = bool(row.get(label, "false"))
                else:
                    data = source_value
                    if isinstance(data, float):
                        data = int(data)
                    json_data[field_name] = str(data)
        
        # json_data["gbl_mdVersion_s"] = "Aardvark" # We don't need this since we define it in the CSV
        return json_data

    # Iterate over each row in the CSV and generate JSON files
    for index, row in csv_data.iterrows():
        json_data = construct_json_data(row)
        
        # Match the existing edu.uwm metadata-aardvark filename convention
        id_part = str(row.get('ID', 'DEFAULT')).strip()
        filename_id = id_part.removeprefix('ark:-77981-')
        if filename_id == id_part:
            filename_id = id_part.split('/')[-1].split('-')[-1]
        file_name = f"{filename_id}_BL_Aardvark.json"
        file_path = os.path.join(output_dir, file_name)
        
        # Write the JSON data to a file
        with open(file_path, 'w', encoding="utf-8") as json_file:
            json.dump(json_data, json_file, indent=4, ensure_ascii=False)

## Step 4: Run the script

In [ ]:
# Convert the CSV to individual JSON files
convert_csv_to_json(csv_file_path, reference_uris_file_path, full_schema_file_path, output_dir)

# Print a message indicating completion
print(f'JSON files generated in directory: {output_dir}')